In [0]:
%pip install openmeteo-requests pvlib pandas pyarrow xgboost scikit-learn seaborn requests
dbutils.library.restartPython()

In [0]:
import openmeteo_requests
import pandas as pd
import numpy as np
import mlflow.sklearn
from pyspark.sql.functions import col, when, expr

# ==========================================
# 1. ARCHITECTURE SETUP & CONFIGURATION
# ==========================================
# Replace this with your latest Run ID from the training pipeline
RUN_ID = "208e8141cc6a4fe39962271efff10bfc" 
MODEL_URI = f"runs:/{RUN_ID}/solar_factor_model"

print("Loading production Weighted Chained Model from MLflow...")
model = mlflow.sklearn.load_model(MODEL_URI)

# ==========================================
# FIXED BRONZE LAYER INGESTION
# ==========================================
# Fetch from the live forecast endpoint
om = openmeteo_requests.Client()
params = {
    "latitude": -31.95,
    "longitude": 115.86,
    "hourly": [
            "cloud_cover_low",   
            "cloud_cover_mid",   
            "cloud_cover_high",  
            "total_column_integrated_water_vapour",
            "sunshine_duration",
            "temperature_2m",        
            "relative_humidity_2m",  
            "surface_pressure"       
        ],
    "timezone": "Australia/Perth",
    "forecast_days": 7  
}
responses = om.weather_api("https://api.open-meteo.com/v1/forecast", params=params)

hourly = responses[0].Hourly()

# Use the API's explicit start, end, and interval values directly
start_epoch = hourly.Time()
end_epoch = hourly.TimeEnd()
step_seconds = hourly.Interval()

# Generate the timestamps explicitly using Pandas native tracking
# This guarantees a true left-to-right 7-day chronological sequence
date_range = pd.date_range(
    start=pd.to_datetime(start_epoch, unit="s"),
    end=pd.to_datetime(end_epoch, unit="s"),
    freq=pd.Timedelta(seconds=step_seconds),
    inclusive="left"
)

# CRITICAL SANITY CHECK: Ensure dimensions match perfectly before Spark conversion
print(f"Generated timestamp vector length: {len(date_range)}")
print(f"Generated weather feature length: {len(hourly.Variables(0).ValuesAsNumpy())}")

pdf_forecast_raw = pd.DataFrame({
    "timestamp": date_range,
    "cloud_low": hourly.Variables(0).ValuesAsNumpy(),
    "cloud_mid": hourly.Variables(1).ValuesAsNumpy(),
    "cloud_high": hourly.Variables(2).ValuesAsNumpy(),
    "water_vapour": hourly.Variables(3).ValuesAsNumpy(),
    "sunshine_duration": hourly.Variables(4).ValuesAsNumpy(),
    "temperature": hourly.Variables(5).ValuesAsNumpy(),
    "relative_humidity": hourly.Variables(6).ValuesAsNumpy(),
    "surface_pressure": hourly.Variables(7).ValuesAsNumpy()
})

# Strip timezone meta-layers to keep Spark's Catalyst optimizer happy
pdf_forecast_raw["timestamp"] = pdf_forecast_raw["timestamp"].dt.tz_localize(None)

# Convert to Spark cleanly
df_forecast_bronze = spark.createDataFrame(pdf_forecast_raw)

# ==========================================
# 3. SILVER LAYER: FEATURE TRANSFORMATION
# ==========================================
print("Engineering features and normalizing arrays...")
# Note: Ensure your pvlib clear-sky geometry logic/UDF is imported/available here
# For the inference matrix, we pass the same transformations used during training

df_forecast_silver = df_forecast_bronze.withColumn(
    "sunshine_fraction", col("sunshine_duration") / 3600.0
).withColumn(
    "cloud_low_fraction", col("cloud_low") / 100.0
).withColumn(
    "cloud_mid_fraction", col("cloud_mid") / 100.0
).withColumn(
    "cloud_high_fraction", col("cloud_high") / 100.0
).withColumn(
    "rh_fraction", col("relative_humidity") / 100.0
)

# Convert features to local Pandas for model execution matrix matching
X_inference_full = df_forecast_silver.toPandas()

# Map exact feature array matrix shape matching the model's signature
feature_cols = [
    "cloud_low_fraction", "cloud_mid_fraction", "cloud_high_fraction",
    "water_vapour", "sunshine_fraction", "temperature", 
    "rh_fraction", "surface_pressure"
]
X_features = X_inference_full[feature_cols].copy()

# Handing NaNs in forecast safely using a forward/backward fill to preserve sequence integrity
X_features = X_features.ffill().bfill()

# ==========================================
# 4. GOLD LAYER: INFERENCE & PHYSICAL OVERRIDES
# ==========================================
print("Generating predictions and executing edge-case business rules...")

# Step A: Base ML Predictions via Chained Model
raw_predictions = model.predict(X_features)
X_inference_full["pred_direct_raw"] = raw_predictions[:, 0]
X_inference_full["pred_diffuse_raw"] = raw_predictions[:, 1]

# Step B: Apply Hard Post-Processing Business Rules & Horizon Overrides
# 1. True Night Rule: If sunshine_fraction is 0, light is physically zero.
# 2. Heavy Overcast Rule: If low cloud deck is 100% thick, clamp direct factors to 0.0.
# 3. Safe Bound Constraint: Ensure outputs are strictly locked between 0.0 and 1.0.

X_inference_full["final_pred_direct"] = np.where(
    X_inference_full["sunshine_fraction"] == 0.0, 0.0, 
    np.where(X_inference_full["cloud_low_fraction"] == 1.0, 0.0, 
             np.clip(X_inference_full["pred_direct_raw"], 0.0, 1.0))
)

X_inference_full["final_pred_diffuse"] = np.where(
    X_inference_full["sunshine_fraction"] == 0.0, 0.0, 
    np.clip(X_inference_full["pred_diffuse_raw"], 0.0, 1.0)
)

# Convert finalized arrays to Spark Gold Layer for production storage/BI dashboarding
df_forecast_gold = spark.createDataFrame(
    X_inference_full[[
        "timestamp", "temperature", "sunshine_fraction", 
        "final_pred_direct", "final_pred_diffuse"
    ]]
)

# Write out to your production Delta lake directory
# df_forecast_gold.write.format("delta").mode("overwrite").saveAsTable("hive_metastore.default.solar_power_14d_forecast")

print("==================================================================")
print("SUCCESS: 7-Day Production Forecast Successfully Written to Gold!")
print(f"Generated {df_forecast_gold.count()} rows of clear sky solar attenuation vectors.")
print("==================================================================")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Bring the dataset locally, sort, and strip timezone structures
pdf_fixed = df_forecast_gold.orderBy("timestamp").toPandas()
pdf_fixed["timestamp"] = pd.to_datetime(pdf_fixed["timestamp"].dt.tz_localize(None))

# Create a clean string column for labels (e.g., "May 26 12:00")
pdf_fixed["time_str"] = pdf_fixed["timestamp"].dt.strftime("%b %d %H:%M")

# 2. Reset the layout - Using an integer range for the X axis to stop the tick error
x_indices = np.arange(len(pdf_fixed))

sns.set_theme(style="whitegrid")
fig, axs = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# ------------------------------------------------------------------
# TOP PLOT: Direct Solar Factor
# ------------------------------------------------------------------
axs[0].plot(x_indices, pdf_fixed["final_pred_direct"], color="#1f77b4", lw=2.5, label="Direct Attenuation Factor")
axs[0].fill_between(x_indices, pdf_fixed["final_pred_direct"], color="#1f77b4", alpha=0.1)
axs[0].set_ylabel("Direct Factor", fontsize=11)
axs[0].set_ylim(-0.05, 1.05)
axs[0].set_title("Operational Solar Yield Forecast Profile (Perth Local Time)", fontsize=13, fontweight="bold")
axs[0].legend(loc="upper right")

# ------------------------------------------------------------------
# MIDDLE PLOT: Diffuse Solar Factor
# ------------------------------------------------------------------
axs[1].plot(x_indices, pdf_fixed["final_pred_diffuse"], color="#ff7f0e", lw=2.5, label="Diffuse Attenuation Factor")
axs[1].fill_between(x_indices, pdf_fixed["final_pred_diffuse"], color="#ff7f0e", alpha=0.1)
axs[1].set_ylabel("Diffuse Factor", fontsize=11)
axs[1].set_ylim(-0.05, 1.05)
axs[1].legend(loc="upper right")

# ------------------------------------------------------------------
# BOTTOM PLOT: Sunshine Fraction
# ------------------------------------------------------------------
axs[2].plot(x_indices, pdf_fixed["sunshine_fraction"], color="#2ca02c", lw=1.5, linestyle="--", label="Sunshine Fraction")
axs[2].set_ylabel("Sunshine Fraction", fontsize=11)
axs[2].set_ylim(-0.05, 1.05)
axs[2].legend(loc="upper right")

# ------------------------------------------------------------------
# MANUAL X-AXIS OVERRIDE: Exactly 7 clean labels over the 7 days
# ------------------------------------------------------------------
# Pick 7 evenly spaced indices across your 336 rows (roughly every 2 days)
tick_indices = np.linspace(0, len(pdf_fixed) - 1, 7, dtype=int)
tick_labels = pdf_fixed["timestamp"].dt.strftime("%b %d").iloc[tick_indices].values

axs[2].set_xticks(tick_indices)
axs[2].set_xticklabels(tick_labels, fontsize=10)

plt.xlabel("7-Day Forecast Window", fontsize=11, labelpad=10)
plt.tight_layout()
plt.show()

In [0]:
import pvlib
from pvlib.location import Location
import pandas as pd
import numpy as np

# 1. Clear Site Geometry
LATITUDE = -31.95
LONGITUDE = 115.86
TZ = "Australia/Perth"
SURFACE_TILT = 25.0     # Ideal tilt for Perth
SURFACE_AZIMUTH = 0.0   # 0 = True North (Southern Hemisphere)
ALBEDO = 0.2           

print("Initializing localized geometry engine...")
site = Location(latitude=LATITUDE, longitude=LONGITUDE, tz=TZ)

# 2. Pull data from Spark and enforce strict Localization
df_gold_predictions = df_forecast_gold.orderBy("timestamp").toPandas()

# CRITICAL FIX: Ensure the index is explicitly localized to Perth AWST for pvlib math
df_gold_predictions["timestamp"] = pd.to_datetime(df_gold_predictions["timestamp"])
if df_gold_predictions["timestamp"].dt.tz is None:
    df_gold_predictions.index = df_gold_predictions["timestamp"].dt.tz_localize(TZ)
else:
    df_gold_predictions.index = df_gold_predictions["timestamp"].dt.tz_convert(TZ)

# 3. Compute precise solar track vectors matching local hours
solar_position = site.get_solarposition(df_gold_predictions.index)
zenith = solar_position['zenith']
apparent_elevation = solar_position['apparent_elevation']
azimuth = solar_position['azimuth']

# 4. Generate the localized clear sky envelopes
clear_sky = site.get_clearsky(df_gold_predictions.index)

# 5. Apply your model's prediction fractions
attenuated_dni = clear_sky['dni'] * df_gold_predictions['final_pred_direct']
attenuated_dhi = clear_sky['dhi'] * df_gold_predictions['final_pred_diffuse']
attenuated_ghi = (attenuated_dni * np.cos(np.radians(zenith))) + attenuated_dhi

# 6. Total Plane-of-Array (POA) GTI calculation
print("Projecting paths onto North-facing planes...")
total_gti = pvlib.irradiance.get_total_irradiance(
    surface_tilt=SURFACE_TILT,
    surface_azimuth=SURFACE_AZIMUTH,
    solar_zenith=zenith,
    solar_azimuth=azimuth,
    dni=attenuated_dni,
    ghi=attenuated_ghi,
    dhi=attenuated_dhi,
    albedo=ALBEDO,
    model='isotropic'
)

# 7. Map variables back to your clean dataframe
df_gold_predictions["gti_total"] = total_gti['poa_global'].values
df_gold_predictions["gti_direct"] = total_gti['poa_direct'].values
df_gold_predictions["gti_diffuse"] = total_gti['poa_diffuse'].values

# Overwrite Gold Spark table with mathematically corrected profiles
df_final_power_forecast = spark.createDataFrame(df_gold_predictions[[
    "timestamp", "temperature", "final_pred_direct", "final_pred_diffuse", 
    "gti_total", "gti_direct", "gti_diffuse"
]])

print("=" * 65)
print("SUCCESS: LOCALIZED GEOMETRIC CORRECTION COMPLETE!")
print(f"Peak predicted 7-day GTI array yield: {df_gold_predictions['gti_total'].max():.2f} W/m²")
print("=" * 65)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

# 1. Pull the computed physics matrix from your final Spark DataFrame
pdf_yield = df_final_power_forecast.orderBy("timestamp").toPandas()
pdf_yield["timestamp"] = pd.to_datetime(pdf_yield["timestamp"].dt.tz_localize(None))

# Create continuous index array to handle layout spacing
x_indices = np.arange(len(pdf_yield))

# 2. Initialize the visualization layout
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 7))

# ------------------------------------------------------------------
# PLOT THE ATTENUATED TILTED PLANE COMPONENTS
# ------------------------------------------------------------------
# Total Plane-of-Array Irradiance (GTI Global)
plt.plot(x_indices, pdf_yield["gti_total"], color="crimson", lw=2.5, 
         label="Total Tilted GTI (Global POA)")
plt.fill_between(x_indices, pdf_yield["gti_total"], color="crimson", alpha=0.08)

# Direct Beam component striking the panel face
plt.plot(x_indices, pdf_yield["gti_direct"], color="gold", lw=1.5, linestyle=":", 
         label="Direct Component on Panel")

# Scattered Sky component striking the panel face
plt.plot(x_indices, pdf_yield["gti_diffuse"], color="deepskyblue", lw=1.5, linestyle="--", 
         label="Diffuse Component on Panel")

# ------------------------------------------------------------------
# X-AXIS TIMELINE LABELLING OVERRIDES (7 Clean Spaced Milestones)
# ------------------------------------------------------------------
tick_indices = np.linspace(0, len(pdf_yield) - 1, 7, dtype=int)
tick_labels = pdf_yield["timestamp"].dt.strftime("%b %d").iloc[tick_indices].values

plt.xticks(tick_indices, tick_labels, fontsize=10)
plt.yticks(fontsize=10)

plt.title("7-Day Global Tilted Irradiance (GTI) Power Yield Time Series Profile (Perth Local Time)", 
          fontsize=13, fontweight='bold', pad=15)
plt.ylabel("Irradiance ($W/m^2$)", fontsize=11, labelpad=10)
plt.xlabel("Forward Forecast Timeline Windows", fontsize=11, labelpad=10)
plt.ylim(-20, pdf_yield["gti_total"].max() * 1.08) # Dynamic padding for peak room
plt.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="none", fontsize=10)

plt.tight_layout()
# current_dir = os.getcwd() 
# save_path = os.path.join(current_dir, '7_Day_GTI_Power_Yield_Profile.png')
# First, create a volume in Catalog Explorer or via SQL:
# CREATE VOLUME IF NOT EXISTS main.default.job_outputs;

save_path = '/Volumes/main/default/solar-figures/7_Day_GTI_Power_Yield_Profile.png'
plt.savefig(save_path, bbox_inches='tight')
plt.savefig(save_path, bbox_inches='tight')
plt.show()

In [0]:
# import base64, requests, os
# from datetime import datetime, UTC

# GITHUB_TOKEN = dbutils.secrets.get(scope="github", key="GITHUB_TOKEN").strip()

# REPO = "jun01ee/solar-yield-forecasting-pipeline"
# BRANCH = "main"
# LOCAL_FILE = save_path
# REMOTE_PATH = "7_Day_GTI_Power_Yield_Profile.png"

# print("repo:", REPO)
# print("branch:", BRANCH)
# print("remote path:", REMOTE_PATH)
# print("file exists:", os.path.exists(LOCAL_FILE))
# print("file size:", os.path.getsize(LOCAL_FILE))
# print("token starts valid:", GITHUB_TOKEN.startswith(("github_pat_", "ghp_")))
# print("token length:", len(GITHUB_TOKEN))

In [0]:
# headers = {
#     "Authorization": f"Bearer {GITHUB_TOKEN}",
#     "Accept": "application/vnd.github+json",
#     "X-GitHub-Api-Version": "2022-11-28",
# }

# r = requests.get("https://api.github.com/user", headers=headers)
# print(r.status_code)
# print(r.text)

In [0]:
# r = requests.get(
#     "https://api.github.com/repos/jun01ee/solar-yield-forecasting-pipeline",
#     headers=headers
# )

# print(r.status_code)
# print(r.json().get("permissions"))
# print(r.json().get("default_branch"))

In [0]:
import os
import shutil
import subprocess
import hashlib
import datetime

GITHUB_TOKEN = dbutils.secrets.get(
    scope="github",
    key="GITHUB_TOKEN"
).strip()

REPO = "jun01ee/solar-yield-forecasting-pipeline"
BRANCH = "main"

# PNG saved from matplotlib
LOCAL_FILE = save_path
# Example:
# "/Volumes/main/default/solar-figures/7_Day_GTI_Power_Yield_Profile.png"

REMOTE_FILE = "7_Day_GTI_Power_Yield_Profile.png"

workdir = "/tmp/solar_repo"

# -------------------------------------------------------------------
# Verify source PNG
# -------------------------------------------------------------------

print("LOCAL_FILE:", LOCAL_FILE)
print("exists:", os.path.exists(LOCAL_FILE))
print("size:", os.path.getsize(LOCAL_FILE))

mtime = os.path.getmtime(LOCAL_FILE)
print(
    "modified:",
    datetime.datetime.fromtimestamp(mtime)
)

with open(LOCAL_FILE, "rb") as f:
    local_hash = hashlib.md5(f.read()).hexdigest()

print("local md5:", local_hash)

# -------------------------------------------------------------------
# Fresh clone
# -------------------------------------------------------------------

if os.path.exists(workdir):
    shutil.rmtree(workdir)

clone_url = (
    f"https://x-access-token:{GITHUB_TOKEN}"
    f"@github.com/{REPO}.git"
)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        BRANCH,
        clone_url,
        workdir
    ],
    check=True
)

# -------------------------------------------------------------------
# Compare existing repo PNG
# -------------------------------------------------------------------

repo_png = os.path.join(workdir, REMOTE_FILE)

if os.path.exists(repo_png):

    with open(repo_png, "rb") as f:
        repo_hash = hashlib.md5(f.read()).hexdigest()

    print("repo md5 :", repo_hash)
    print("same file:", local_hash == repo_hash)

# -------------------------------------------------------------------
# Replace PNG in cloned repo
# -------------------------------------------------------------------

shutil.copyfile(
    LOCAL_FILE,
    repo_png
)

# -------------------------------------------------------------------
# Git config
# -------------------------------------------------------------------

subprocess.run(
    ["git", "config", "user.name", "Juno Li"],
    cwd=workdir,
    check=True
)

subprocess.run(
    ["git", "config", "user.email", "juno.li.research@gmail.com"],
    cwd=workdir,
    check=True
)

# -------------------------------------------------------------------
# Commit + push
# -------------------------------------------------------------------

subprocess.run(
    ["git", "add", REMOTE_FILE],
    cwd=workdir,
    check=True
)

commit_result = subprocess.run(
    ["git", "commit", "-m", "Update daily plot"],
    cwd=workdir,
    text=True,
    capture_output=True
)

print(commit_result.stdout)
print(commit_result.stderr)

if commit_result.returncode == 0:

    subprocess.run(
        ["git", "push", "origin", BRANCH],
        cwd=workdir,
        check=True
    )

    print("PNG pushed to GitHub")

else:
    print("No changes to commit")